In [12]:
import pandas as pd
import numpy as np

### Funciones

In [13]:
def calcular_error_medicion(escala, vpp):
    """
    Calcula el error de medición en función de la escala de medición (Volts/División).

    Parámetros:
    escala (pd.Series): Serie con las escalas de medición.
    vpp (pd.Series): Serie con los valores de Vpp.

    Retorna:
    pd.Series: Serie con los errores de medición calculados.
    """

    # 1. Definimos las condiciones (en Voltios)
    condiciones = [
        (escala >= 0.002) & (escala <= 0.005),  # 2mV a 5mV -> 4%
        (escala >= 0.010) & (escala <= 10.0),  # 10mV a 10V -> 3%
    ]
    # 2. Definimos los porcentajes correspondientes
    porcentajes = [0.04, 0.03]
    # 3. Asignamos el porcentaje según la escala
    pct_aplicable = np.select(condiciones, porcentajes, default=np.nan)
    # 4. Calculamos el error de medición
    error_vpp = vpp * pct_aplicable
    return error_vpp


In [14]:
def redondear_cifras_significativas(valor, n):
    """
    Redondea un valor a un número específico de cifras significativas.
    """
    if pd.isna(valor) or valor == 0:
        return valor
    # Calcula los decimales necesarios según la magnitud del número
    decimales = n - 1 - int(np.floor(np.log10(abs(valor))))
    return round(valor, decimales)

In [15]:
def redondear_medicion(row, valor_verdadero, incerteza):
    """
    Redondea una medición según su incertidumbre.
    """
    medicion = row[valor_verdadero]
    incertidumbre = row[incerteza]

    if pd.isna(medicion) or pd.isna(incertidumbre) or incertidumbre == 0:
        return medicion

    # Determina a qué posición decimal debemos redondear
    # Ejemplo: para 0.3 -> -log10(0.3) = 0.52 -> floor = 0 -> decimales = 1
    # Ejemplo: para 0.04 -> -log10(0.04) = 1.39 -> floor = 1 -> decimales = 2
    decimales = int(-np.floor(np.log10(abs(incertidumbre))))

    if decimales >= 0:
        return round(medicion, decimales)
    else:
        # Si la incertidumbre está en las decenas/centenas (ej: ±30)
        return round(medicion, decimales)

### Importamos los datos

In [16]:
path = r".\clase2\raw\mediciones_parte1_raw.csv"

In [25]:
df = pd.read_csv(path, sep=";")
df.head(5)

,Frecuencia (Khz),Vpp-Emisor (V),Vpp-Receptor (mV),Escala-receptor (mV)
0,35.0,7.12,10,50
1,36.0,7.12,32,50
2,37.0,7.12,74,50
3,38.0,7.12,188,50
4,38.1,7.12,202,50


In [26]:
df.shape

(47, 4)

In [27]:
#df = df.drop(columns=["Unnamed: 8"])

In [28]:
df = df.dropna()
df.shape

(47, 4)

In [29]:
df.columns

Index(['Frecuencia (Khz)', 'Vpp-Emisor (V)', 'Vpp-Receptor (mV)',
       'Escala-receptor (mV)'],
      dtype='str')

### Limpiando datos

In [32]:
df_clean = pd.DataFrame()
df_clean["frecuencia"] = df["Frecuencia (Khz)"]

In [44]:
528*0.03

15.84

In [39]:
df["Vpp-Receptor (mV)"]

0      10
1      32
2      74
3     188
4     202
5     214
6     232
7     256
8     292
9     324
10    372
11    448
12    528
13    640
14    736
15    816
16    848
17    856
18    840
19    816
20    808
21    800
22    784
23    744
24    704
25    616
26    496
27    384
28    296
29    224
30    180
31    148
32    120
33    102
34     86
35     72
36     60
37     49
38     42
39     36
40     32
41     30
42     29
43     33
44     28
45     17
46     14
Name: Vpp-Receptor (mV), dtype: int64

In [38]:
df_clean["vpp"] = df["Vpp-Receptor (mV)"]
df_clean["vpp_error"] = calcular_error_medicion(df["Escala-receptor (mV)"] * 0.001, df["Vpp-Receptor (mV)"])
df_clean["vpp_error"] = df_clean["vpp_error"].apply(lambda x: redondear_cifras_significativas(x, 1))
df_clean["vpp"] = df_clean.apply(lambda row: redondear_medicion(row, "vpp", "vpp_error"), axis=1)
df_clean

,frecuencia,vpp,vpp_error
0,35.0,10.0,0.3
1,36.0,32.0,1.0
2,37.0,74.0,2.0
3,38.0,188.0,6.0
4,38.1,202.0,6.0
5,38.2,214.0,6.0
6,38.3,232.0,7.0
7,38.4,256.0,8.0
8,38.5,292.0,9.0
9,38.6,320.0,10.0


In [45]:
#vt_error = np.array([0.01]*10)
#vt_error

In [38]:
name_columns = ["10 Hz", "100 Hz", "1 KHz", "10 KHz", "100 KHz", "1 MHz"]

In [45]:
for i, name in enumerate(name_columns, start=1):
    df_clean[name] = df.iloc[:, i:i+1]

In [47]:
df_clean["vt_error"] = vt_error

In [48]:
df_clean

,vpp,vpp_error,10 Hz,100 Hz,1 KHz,10 KHz,100 KHz,1 MHz,vt_error
0,2.04,0.06,0.43,0.63,0.60,0.43,0.02,0.010,0.01
1,4.00,0.10,1.25,1.34,1.28,1.54,0.24,0.003,0.01
2,6.00,0.20,2.04,2.11,2.01,2.73,1.06,0.003,0.01
3,8.00,0.20,2.82,2.79,2.68,3.91,1.61,0.002,0.01
4,10.00,0.30,3.58,3.54,3.43,5.03,1.87,0.002,0.01
5,12.00,0.40,4.34,4.24,4.13,6.24,1.96,0.002,0.01
6,14.10,0.40,5.15,4.98,4.87,7.42,2.01,0.002,0.01
7,16.00,0.50,5.87,5.53,5.42,8.32,2.04,0.002,0.01
8,18.00,0.50,6.47,6.24,6.09,9.58,2.06,0.001,0.01
9,20.00,0.60,7.18,6.97,6.85,10.83,2.07,0.001,0.01


### Guardando los datos

In [46]:
path_clean = r".\clase2\clean\mediciones_parte1_clean.csv"

In [47]:
df_clean.to_csv(path_clean, index=False)